# W13B · 캠퍼스 리폼 전시 안내 키트

**11월 25일 · 개인 CPU 실습 · 학습 → 검증 → 내 소개문 HTML**

먼저 체험용 ZIP의 index.html을 엽니다. 앱에서 운영 메모와 모델 원문, 실행 ID를 찾으세요. 오늘은 토트백·운동화·티셔츠의 작품 소개문을 만듭니다. 11/30에는 같은 작품에 포스터를 붙입니다.

Colab에서 **파일 → Drive에 사본 저장** 후 CPU로 시작합니다. 첫 모델은 약 1GB를 내려받습니다. 라이브가 멈추면 런타임을 다시 연결하고 00으로 돌아옵니다.


In [ ]:
PACKAGE_BYTES = '...'
import base64, io, os, sys, tempfile, zipfile
from pathlib import Path
from IPython.display import HTML, FileLink
from IPython.display import display

ON_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
if ON_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==5.15.1", "peft==0.21.2", "ipywidgets>=8,<9"], check=True)
work = Path(tempfile.mkdtemp(prefix="w13b-exhibit-"))
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PACKAGE_BYTES))) as archive:
    archive.extractall(work)
sys.path.insert(0, str(work))
from luna_genai.exhibit_studio import (dataset, ExhibitTrainer, FIELDS, ROOT, STEPS,
                                      RANKS, summarize, select, load_json, bundle, html_for)


## 01 · 데이터와 손실을 연결하기

운영팀이 홀수/짝수로 임의로 나눈 것이 아닙니다. 같은 작품의 문장 변형과 버전은 같은 작품 ID에 묶입니다. 훈련 24건의 목표 답으로만 가중치를 바꾸고, 검증 9건은 후보를 고르는 데 씁니다. 교수자가 가진 최종 평가 9건은 나중에 엽니다.

Qwen2의 한 칸 시프트에서 현재 위치의 로짓은 다음 토큰을 예측합니다. 정답 확률이 0.5라면 $-\log(0.5)\simeq0.693$입니다. 프롬프트는 읽되 손실의 라벨에서 $-100$으로 마스킹합니다.

[W03B 다음 토큰과 한 단계 앱](https://github.com/lunalab-ai/genAI/blob/2026-fall-w03b/course/notion/w03b/w03b-transformer-walkthrough.md#L174) · [W10A 손실 마스크](https://github.com/lunalab-ai/genAI/blob/2026-fall-w10a/course/notion/w10a/w10a-lesson.md#L203)


In [ ]:
cases = dataset()
train = [case for case in cases if case["split"] == "train"]
valid = [case for case in cases if case["split"] == "validation"]
print(len(train), len(valid), train[0]["id"], valid[0]["id"])
print(train[0]["title"], train[0]["facts"], train[0]["target"], sep="\n")


**입력 → 호출 → 반환**. 예시 prefix는 [11, 12], response는 [21, 22]입니다. 호출에서 앞의 두 자리만 무시로 바꾸고 뒤의 두 정답은 보존합니다.

**힌트 1** 앞부분은 채점 대상에서 빠집니다. **힌트 2** 앞부분 길이는 len(prefix)입니다. **힌트 3** 같은 값의 리스트를 길이만큼 반복하고 response를 뒤에 붙이세요.

**정상 출력** 앞의 두 칸은 손실 제외, 뒤에는 21과 22. 마스크가 틀리면 수업 실측 재생이 열립니다. 줄을 수정하고 여기부터 아래 셀을 실행하세요.


In [ ]:
def make_labels(prefix, response):
    prefix_labels = list(prefix)  # TODO: 앞부분의 토큰마다 손실 제외 표시를 넣으세요.
    return prefix_labels + list(response)


In [ ]:
trial = make_labels([11, 12], [21, 22])
ready = trial == [-100, -100, 21, 22]
print(trial, "손실 마스크 연결" if ready else "TODO 확인 · 수업 실측 재생")
OFFLINE_REPLAY = False  # @param {type:"boolean"}
if OFFLINE_REPLAY:
    print("오프라인 수업 실측 재생 · 현재 모델을 불러오지 않습니다.")
else:
    trainer = ExhibitTrainer()
    tok = trainer.tokens(train[0])
    print("실제 배열 길이", len(tok["prefix"]), len(tok["response"]))
    print("경계 토큰", tok["prefix"][-4:], tok["response"][:4])


## 02 · LoRA 두 후보를 같은 훈련 자료로 학습

$W'=W+(\alpha/r)BA$. 16×16 예에서 rank 2와 4는 64개와 128개를 학습합니다. 실제 모델 모수 개수는 아래 표에 나옵니다. rank 외의 설정을 고정합니다. 24건 24회, LR 0.0005, seed 42, $\alpha/r=2$, q_proj와 v_proj.

먼저 두 설정에서 바뀔 모수 수와 자료 밖 값이 어느 조건에서 줄지 예측하세요. 학습 손실, 기본 가중치의 해시, 어댑터의 실제 변화 수를 확인합니다.

[W10A LoRA와 차원](https://github.com/lunalab-ai/genAI/blob/2026-fall-w10a/course/notion/w10a/w10a-lesson.md#L30)


In [ ]:
reference = load_json(ROOT / "reference.json")
training = {}
evidence = {}
# 수업에서는 3, 과제 제출에서는 9로 바꾸세요.
EVAL_COUNT = 3  # @param [3, 9] {type:"raw"}
FORCE_REPLAY = False  # @param {type:"boolean"}
live = ready and not FORCE_REPLAY and not OFFLINE_REPLAY
if live:
    for rank in RANKS:
        print("실제 학습 시작", rank, flush=True)
        row = trainer.fit(train, work / f"rank{rank}", rank=rank, steps=STEPS, mask=make_labels)
        training[str(rank)] = row
        print(rank, "처음·마지막 손실", row["updates"][0]["loss"], row["updates"][-1]["loss"],
              "학습 모수", row["trainable"], "변경 텐서", row["changed_tensors"], flush=True)
        evidence[str(rank)] = trainer.evaluate(valid[:EVAL_COUNT], rank)
else:
    training = reference["training"]
    evidence = {str(rank): reference["evaluation"][str(rank)][:EVAL_COUNT] for rank in RANKS}
    # 실제 모델 경로는 시작본에서도 한 건 확인합니다. 뒤의 표는 제공한 실측의 재생입니다.
    if not OFFLINE_REPLAY:
        smoke = trainer.generate_case(valid[0], 0)
        assert smoke["result"]["raw"]
        print("기본 실제 생성", smoke["run_id"], "· 아래 비교는 수업 실측 재생")
mode = "live" if live else "replay"


## 03 · 학습 손실과 실제 앱의 사실을 함께 읽기

정답 토큰을 넣은 NLL은 각 토큰 손실 합을 전체 정답 토큰 수로 나눕니다. 새로 생성한 원문은 종류·장소·관람료·예약을 운영 자료와 대조합니다. 자료에 없는 예약은 확인 필요가 참조값입니다. 오류가 있는 후보는 표에서 확인하고, 검증에서 자료 밖 값과 형식 오류가 없는 후보를 우선 적용합니다. 이런 후보가 없으면 앱을 게시 보류 상태로 시연합니다.

[W10A 평가와 분모](https://github.com/lunalab-ai/genAI/blob/2026-fall-w10a/course/notion/w10a/w10a-lesson.md#L220)


In [ ]:
import pandas as pd
comparison = [summarize(evidence[str(rank)]) for rank in RANKS]
display(pd.DataFrame(comparison))
selected = select(comparison)
print("배포 후보", selected if selected is not None else "게시 보류")
print("실행 모드", "실제 학습" if live else "수업 실측 재생")


## 04 · 웹 앱과 연결하기

**[선택한 모델로 소개문 생성]**이 현재 설정의 어댑터를 불러와 같은 추론 함수를 호출합니다. 모델 원문과 검증표를 먼저 확인하고 HTML을 받습니다. 별도 서버는 필요 없습니다. 다운로드한 index.html을 브라우저에서 열면 같은 실행 ID가 나타납니다.


In [ ]:
rank_to_apply = selected if selected is not None else 2
case = valid[0]
out = work / "index.html"
submit = work / "w13b-assignment1.zip"

def apply_model(_button=None):
    global app_run
    app_run = (trainer.generate_case(case, rank_to_apply) if live else
               next(row for row in evidence[str(rank_to_apply)] if row["case_id"] == case["id"]))
    out.write_bytes(html_for(case, app_run, comparison, mode).encode("utf-8"))
    bundle(submit, training, evidence, case, app_run, mode)
    print("실행 ID", app_run["run_id"], "· rank", app_run["rank"])
    print(app_run["result"]["raw"])
    display(pd.DataFrame(app_run["result"]["fields"]))
    return app_run

# 검증 실행도 버튼에 연결된 동일 함수를 호출합니다.
apply_model()
if ON_COLAB:
    import ipywidgets as widgets
    from google.colab import files
    screen = widgets.Output()
    run_button = widgets.Button(description="선택한 모델로 소개문 생성")
    def run_click(button):
        with screen:
            screen.clear_output()
            apply_model(button)
    run_button.on_click(run_click)
    html_button = widgets.Button(description="내 키트 HTML 받기")
    html_button.on_click(lambda _: files.download(str(out)))
    zip_button = widgets.Button(description="과제 ZIP 받기")
    zip_button.on_click(lambda _: files.download(str(submit)))
    display(widgets.VBox([run_button, html_button, zip_button, screen]))


**새 과제 1 · 12월 5일 토요일 23:59 KST.** 02에서 EVAL_COUNT를 9로 바꾸고 02–04를 실행합니다. ZIP에는 학습 로그, 원문, 판정표와 앱이 들어갑니다. 수업 예시와 내 실행에서 지표를 읽는 규칙은 같습니다.


In [ ]:
assert len(train) == 24 and len(valid) == 9
assert all(len(training[str(r)]["updates"]) == 24 for r in RANKS)
assert all(training[str(r)]["base_before"] == training[str(r)]["base_after"] and
           training[str(r)]["changed_tensors"] > 0 for r in RANKS)
assert app_run["result"]["raw"] and app_run["run_id"] in out.read_text(encoding="utf-8")
with zipfile.ZipFile(submit) as archive:
    assert archive.read("index.html") == out.read_bytes()
print("W13B_RUNTIME_PASS", mode, app_run["run_id"], submit)
